In [11]:
from huggingface_hub import hf_hub_download
import pandas as pd
from pathlib import Path

In [12]:
REPO_ID = "pyvene/axbench-concept10"
SUBSET = "9b/l20"
STORE = Path("../store/concept10")
STORE.mkdir(parents=True, exist_ok=True)

In [13]:
# Use axbench's official train/test parquets for one model/layer subset.
# Train holds the main concept per concept_id (all positive).
# Test holds the main concept (positive + negative) + distractor concepts at
# the same concept_id slot; we drop distractors by matching the main concept
# name from train, then keep positives only for column consistency.

def load_split(split: str) -> pd.DataFrame:
    path = hf_hub_download(REPO_ID, f"{SUBSET}/{split}/data.parquet", repo_type="dataset")
    return pd.read_parquet(path)

tr = load_split("train")
te = load_split("test")

main = (
    tr[tr["concept_id"] >= 0]
    .groupby("concept_id")["output_concept"].first().to_dict()
)

def to_schema(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df["concept_id"] >= 0].copy()
    df = df[df.apply(lambda r: r["output_concept"] == main[r["concept_id"]], axis=1)]
    df = df[df["category"] == "positive"]
    return pd.DataFrame({
        "concept":  df["output_concept"].values,
        "question": df["input"].values,
        "answer":   df["output"].values,
    })

train_df = to_schema(tr)
test_df  = to_schema(te)

train_df.to_csv(STORE / "train.csv", index=False)
test_df.to_csv(STORE / "test.csv", index=False)

def stats(df):
    by = df.groupby("concept").size()
    return {"rows": len(df), "concepts": by.shape[0], "rows/concept (mean)": round(by.mean(), 1), "min-max": f"{by.min()}-{by.max()}"}

print(pd.DataFrame({"train": stats(train_df), "test": stats(test_df)}).T)
print(f"\ntrain/test ratio: {len(train_df)/len(test_df):.2f}")

      rows concepts rows/concept (mean) min-max
train  720       10                72.0   72-72
test   360       10                36.0   36-36

train/test ratio: 2.00


In [14]:
print("concepts:")
for c in sorted(train_df["concept"].unique()):
    print(" ", c)

concepts:
  control flow statements and conditional logic in programming
  functional aspects related to processes and operations in a system
  phrases related to biological interactions and measurements
  programming constructs and data structures in code snippets
  references to default settings or configurations
  references to nighttime or nocturnal themes
  references to payment, salaries, and financial arrangements
  specific names and geographical locations, particularly related to legal cases or contexts
  terms related to biochemical compounds and their effects
  terms related to online gambling and casinos
